# 数理変換タクティクス AI トレーニング（7操作対応版）

| カテゴリ | 操作 | 概要 |
|---|---|---|
| 0 | パス | ターン終了 |
| 1 | 約数強奪攻撃 | 自分のカードの倍数を相手から奪う |
| 2 | 足し算 | 2枚 → 合計 |
| 3 | 引き算 | 2枚 → 差（正のみ） |
| 4 | 商×余 | 2枚 → floor(a/b) × (a%b) |
| 5 | 桁和で分裂 | 1枚 → 桁和で割った商と余 |
| 6 | GCD掛け算 | 2枚 → X × gcd(X,Y) |

In [ ]:
!pip install tensorflowjs scikit-learn -q
print('インストール完了！')

In [ ]:
import numpy as np
import tensorflow as tf
import tensorflowjs as tfjs
import random
import math
from sklearn.model_selection import train_test_split

print(f'TensorFlow: {tf.__version__}')

NUM_ACTIONS = 7  # 全操作数

## ゲームシミュレーター（全7操作対応）

In [ ]:
def gcd(a, b):
    while b:
        a, b = b, a % b
    return a

def digit_sum(n):
    return sum(int(d) for d in str(n))

class MathCardGame:
    """
    アクションカテゴリ:
      0 = パス（ターン終了）
      1 = 約数強奪攻撃
      2 = 足し算（2枚 → a+b）
      3 = 引き算（2枚 → a-b > 0）
      4 = 商×余（2枚 → floor(a/b) * (a%b)）
      5 = 桁和で分裂（1枚 → q と r）
      6 = GCD掛け算（2枚 → X * gcd(X,Y)）
    """
    def __init__(self, config=None):
        self.config = config or {
            'initHandCount': 5,
            'initMaxNum': 10,
            'drawCount': 2,
            'drawMaxNum': 20,
            'handLimitNum': 150,
            'winScore': 100,
            'maxTurns': 10,
        }

    def reset(self):
        cfg = self.config
        self.hands = {
            'P1': [random.randint(1, cfg['initMaxNum']) for _ in range(cfg['initHandCount'])],
            'P2': [random.randint(1, cfg['initMaxNum']) for _ in range(cfg['initHandCount'])],
        }
        self.scores = {'P1': 0, 'P2': 0}
        self.turn_count = 1
        self.current_player = 'P1'
        self.done = False
        self.winner = None

    def opponent(self, pid):
        return 'P2' if pid == 'P1' else 'P1'

    def get_state_vector(self, player_id):
        """30次元ベクトル（cpu-ai.jsと同じ形式）"""
        vector = [0.0] * 30
        my_hand = self.hands[player_id]
        enemy_hand = self.hands[self.opponent(player_id)]
        for i, c in enumerate(my_hand[:10]):
            vector[i] = c / 150.0
        vector[10] = self.scores[player_id] / 100.0
        for i, c in enumerate(enemy_hand[:10]):
            vector[11 + i] = c / 150.0
        vector[21] = self.scores[self.opponent(player_id)] / 100.0
        vector[22] = self.turn_count / 10.0
        return vector

    # ─── 各操作の「最善候補」を返すヘルパー ───

    def best_attack(self, pid):
        """(card_idx, atk_num, gain) or None"""
        enemy = self.hands[self.opponent(pid)]
        best, best_gain = None, 0
        for i, atk in enumerate(self.hands[pid]):
            if atk == 1: continue
            gain = sum(n for n in enemy if n % atk == 0)
            if gain > best_gain:
                best_gain = gain
                best = (i, atk, gain)
        return best

    def best_add(self, pid):
        """(i, j, result) or None  ― a+b 最大"""
        hand = self.hands[pid]
        limit = self.config['handLimitNum']
        best, best_val = None, -1
        for i in range(len(hand)):
            for j in range(len(hand)):
                if i == j: continue
                r = hand[i] + hand[j]
                if r <= limit and r > best_val:
                    best_val = r; best = (i, j, r)
        return best

    def best_sub(self, pid):
        """(i, j, result) or None  ― hand[i]-hand[j] > 0 最大"""
        hand = self.hands[pid]
        best, best_val = None, 0
        for i in range(len(hand)):
            for j in range(len(hand)):
                if i == j: continue
                r = hand[i] - hand[j]
                if r > 0 and r > best_val:
                    best_val = r; best = (i, j, r)
        return best

    def best_divprod(self, pid):
        """(i, j, result) or None  ― floor(a/b)*(a%b) 最大"""
        hand = self.hands[pid]
        limit = self.config['handLimitNum']
        best, best_val = None, -1
        for i in range(len(hand)):
            for j in range(len(hand)):
                if i == j or hand[j] == 0: continue
                q, r = divmod(hand[i], hand[j])
                res = q * r
                if res > 0 and res <= limit and res > best_val:
                    best_val = res; best = (i, j, res)
        return best

    def best_digitsumDiv(self, pid):
        """(idx, q, r) or None  ― 桁和で割り商が最大"""
        hand = self.hands[pid]
        limit = self.config['handLimitNum']
        best, best_val = None, -1
        for i, num in enumerate(hand):
            ds = digit_sum(num)
            if ds == 0: continue
            q, r = divmod(num, ds)
            if q <= limit and q > best_val:
                best_val = q; best = (i, q, r)
        return best

    def best_gcdMul(self, pid):
        """(i, j, result) or None  ― X*gcd(X,Y) 最大"""
        hand = self.hands[pid]
        limit = self.config['handLimitNum']
        best, best_val = None, -1
        for i in range(len(hand)):
            for j in range(len(hand)):
                if i == j: continue
                g = gcd(hand[i], hand[j])
                if g <= 1: continue
                res = hand[i] * g
                if res <= limit and res > best_val:
                    best_val = res; best = (i, j, res)
        return best

    # ─── ルールベース最適行動 ───

    def get_optimal_action(self, pid):
        my_score = self.scores[pid]
        win = self.config['winScore']
        atk = self.best_attack(pid)
        add = self.best_add(pid)
        sub = self.best_sub(pid)
        dp  = self.best_divprod(pid)
        dsd = self.best_digitsumDiv(pid)
        gm  = self.best_gcdMul(pid)

        # 攻撃で即勝利できるなら最優先
        if atk and self.turn_count > 1 and my_score + atk[2] >= win:
            return 1

        # GCDで大きな数を作れるなら（将来の攻撃力UP）
        if gm and gm[2] >= 30:
            return 6

        # 攻撃が有効（相手から10点以上奪える）
        if atk and self.turn_count > 1 and atk[2] >= 10:
            return 1

        # 足し算で大きな数ができる
        if add and add[2] >= 20:
            return 2

        # 商×余で有用な数ができる
        if dp and dp[2] >= 15:
            return 4

        # 攻撃が少しでもある
        if atk and self.turn_count > 1:
            return 1

        # GCDで少し大きな数
        if gm and gm[2] >= 10:
            return 6

        # 桁和で分裂（数を増やす）
        if dsd and dsd[1] >= 5:
            return 5

        # とにかく足し算
        if add:
            return 2

        # 商×余
        if dp:
            return 4

        # 引き算（差が正のとき）
        if sub:
            return 3

        return 0  # パス

    # ─── 行動実行（ターン終了まで含む） ───

    def _end_turn(self, pid):
        self.current_player = self.opponent(pid)
        if self.current_player == 'P1':
            self.turn_count += 1
            if self.turn_count > self.config['maxTurns']:
                self.done = True
                self.winner = max(self.scores, key=lambda p: self.scores[p])

    def execute_action(self, pid, action):
        hand = self.hands[pid]
        limit = self.config['handLimitNum']
        reward = 0

        if action == 1:  # 攻撃
            atk = self.best_attack(pid)
            if atk:
                i, atk_num, gain = atk
                eid = self.opponent(pid)
                self.hands[eid] = [n for n in self.hands[eid] if n % atk_num != 0]
                hand.pop(i)
                self.scores[pid] += gain
                reward = gain
                if self.scores[pid] >= self.config['winScore']:
                    self.done = True; self.winner = pid; reward += 50

        elif action == 2:  # 足し算
            add = self.best_add(pid)
            if add:
                i, j, res = add
                hand = [c for k, c in enumerate(hand) if k != i and k != j]
                hand.append(res)
                reward = res / 50.0

        elif action == 3:  # 引き算
            sub = self.best_sub(pid)
            if sub:
                i, j, res = sub
                hand = [c for k, c in enumerate(hand) if k != i and k != j]
                hand.append(res)
                reward = res / 50.0

        elif action == 4:  # 商×余
            dp = self.best_divprod(pid)
            if dp:
                i, j, res = dp
                hand = [c for k, c in enumerate(hand) if k != i and k != j]
                hand.append(res)
                reward = res / 50.0

        elif action == 5:  # 桁和で分裂
            dsd = self.best_digitsumDiv(pid)
            if dsd:
                idx, q, r = dsd
                hand.pop(idx)
                hand.append(q)
                if r > 0: hand.append(r)
                reward = q / 50.0

        elif action == 6:  # GCD掛け算
            gm = self.best_gcdMul(pid)
            if gm:
                i, j, res = gm
                hand = [c for k, c in enumerate(hand) if k != i and k != j]
                hand.append(res)
                reward = res / 50.0

        self.hands[pid] = hand
        self._end_turn(pid)
        return reward

print('ゲームシミュレーター準備完了！（7操作対応）')

## トレーニングデータ生成

In [ ]:
NUM_EPISODES = 30000

X_data, y_data = [], []
game = MathCardGame()
wins = {}

for ep in range(NUM_EPISODES):
    game.reset()
    while not game.done:
        pid = game.current_player
        X_data.append(game.get_state_vector(pid))
        action = game.get_optimal_action(pid)
        y_data.append(action)
        game.execute_action(pid, action)
    if game.winner:
        wins[game.winner] = wins.get(game.winner, 0) + 1
    if (ep + 1) % 10000 == 0:
        print(f'エピソード {ep+1}/{NUM_EPISODES} 完了 ... データ数: {len(X_data)}')

X_data = np.array(X_data, dtype=np.float32)
y_data = np.array(y_data, dtype=np.int32)

print(f'\n生成完了！総データ数: {len(X_data)}')
action_names = ['パス', '攻撃', '足し算', '引き算', '商×余', '桁和分裂', 'GCD掛け']
unique, counts = np.unique(y_data, return_counts=True)
for u, c in zip(unique, counts):
    print(f'  {u}({action_names[u]}): {c}件 ({c/len(y_data)*100:.1f}%)')

## ニューラルネットワーク トレーニング（7クラス）

In [ ]:
from tensorflow.keras.utils import to_categorical

X_train, X_val, y_train, y_val = train_test_split(
    X_data, y_data, test_size=0.1, random_state=42
)
y_train_cat = to_categorical(y_train, num_classes=NUM_ACTIONS)
y_val_cat   = to_categorical(y_val,   num_classes=NUM_ACTIONS)

model = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation='relu', input_shape=(30,)),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(NUM_ACTIONS, activation='softmax'),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)
]

history = model.fit(
    X_train, y_train_cat,
    validation_data=(X_val, y_val_cat),
    epochs=60, batch_size=256,
    callbacks=callbacks, verbose=1
)
print(f'\n最高検証精度: {max(history.history["val_accuracy"])*100:.1f}%')

In [ ]:
import matplotlib.pyplot as plt
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['accuracy'], label='訓練')
ax1.plot(history.history['val_accuracy'], label='検証')
ax1.set_title('精度'); ax1.legend()
ax2.plot(history.history['loss'], label='訓練')
ax2.plot(history.history['val_loss'], label='検証')
ax2.set_title('損失'); ax2.legend()
plt.tight_layout(); plt.show()

## TensorFlow.js 形式でエクスポート

In [ ]:
import os, shutil
OUT = 'tfjs_model'
if os.path.exists(OUT): shutil.rmtree(OUT)
tfjs.converters.save_keras_model(model, OUT)
print('エクスポート完了！')
for f in os.listdir(OUT):
    print(f'  {f} ({os.path.getsize(os.path.join(OUT,f))/1024:.1f} KB)')

In [ ]:
shutil.make_archive('tfjs_model', 'zip', 'tfjs_model')
from google.colab import files
files.download('tfjs_model.zip')
print('ダウンロード開始！')
print('ZIPを解凍して game-/tfjs_model/ に入れてgit pushしてください。')

## 動作テスト

In [ ]:
action_names = ['パス', '攻撃', '足し算', '引き算', '商×余', '桁和分裂', 'GCD掛け']

tests = [
    {'p1': [6, 3, 10, 7, 2], 'p2': [12, 18, 5, 9, 4], 'desc': '攻撃チャンスあり（6は12,18の約数）'},
    {'p1': [5, 8, 3, 2, 7],  'p2': [1, 1, 1, 1, 1],  'desc': '合成が有利（相手が全部1）'},
    {'p1': [6, 9, 4, 3, 2],  'p2': [8, 10, 6, 4, 3], 'desc': 'GCD有効（6と9のGCDは3）'},
]

for t in tests:
    g = MathCardGame()
    g.reset()
    g.hands['P1'] = t['p1']
    g.hands['P2'] = t['p2']
    g.turn_count = 3
    g.scores = {'P1': 20, 'P2': 30}
    vec = np.array([g.get_state_vector('P1')], dtype=np.float32)
    pred = model.predict(vec, verbose=0)[0]
    chosen = int(np.argmax(pred))
    optimal = g.get_optimal_action('P1')
    print(f'【{t["desc"]}】')
    print(f'  自分: {t["p1"]}  相手: {t["p2"]}')
    print(f'  AI選択: {action_names[chosen]} | 最適: {action_names[optimal]} | {"✅" if chosen==optimal else "❌"}')
    probs = '  '.join(f'{action_names[i]}:{pred[i]:.2f}' for i in range(NUM_ACTIONS))
    print(f'  確率: {probs}')
    print()